### Day 19 · 数据摄取(CSV / Excel / JSON / SQL / API)**pandas** 提供 `read_csv`、`read_excel`、`read_json`、`read_sql`从多种数据源读入数据。真实工作中数据来自文件、数据库、网络 API,必须掌握每种来源的读取方式和常见坑(编码、分隔符、类型推断)。前置:Day11-14 已掌握 Python 核心 / NumPy / Pandas 基础。

#### read_csv(文件路径 / 编码 / 分隔符)痛点:拿到一份 CSV 文件直接 `pd.read_csv("data.csv")` 经常报错:中文乱码、分隔符不是逗号、首行不是列名。类比:读 CSV 像开快递箱 -- `encoding` 是语言(中英)、`sep` 是封口方式(胶带/封条)、`header` 是第几行才是说明书。解释:`read_csv` 最常用,核心参数: `encoding` 指定字符编码(中文常用utf-8 或 gbk)、`sep` 指定列分隔符(默认逗号)、`header` 指定第几行作为列名(默认 0)。

In [ ]:
import pandas as pd# 构造示例数据,写出 CSV 文件df = pd.DataFrame({    "姓名": ["张三", "李四", "王五", "赵六"],    "年龄": [20, 21, 19, 22],    "成绩": [88.5, 92.0, 76.5, 81.0],})df.to_csv("students.csv", index=False, encoding="utf-8")# --- 执行过程 ---# 第 1 行 df = pd.DataFrame({...}):#   ① 在内存中创建 DataFrame,4 行 3 列# 第 2 行 df.to_csv(..., index=False, encoding='utf-8'):#   ① 把 DataFrame 写到磁盘,index=False 不导出行号#   ② 文件内容是 utf-8 编码的中文+数字df_read = pd.read_csv("students.csv", encoding="utf-8")# 第 3 行 pd.read_csv:#   ① 按 utf-8 编码打开文件#   ② 第 0 行识别为列名,其余行是数据print("--- 基本读取 ---")print(df_read)

> **逐行解剖**> - `encoding="utf-8"`:按 utf-8 读文件。中文 CSV 常见 utf-8 或 gbk,>   读错就乱码。> - `index=False`(to_csv 参数):写出时不含默认行索引 0,1,2,读回更干净。

> **常见错误**> 1. **错误现象**:`UnicodeDecodeError: 'utf-8' codec can't decode >    byte 0xa1`>    **原因**:文件实际是 gbk 却指定 encoding="utf-8",改 encoding="gbk">   即可。> 2. **错误现象**:列名变成 "姓名;年龄;成绩",数据全挤在一列>    **原因**:文件实际用分号分隔,未指定 `sep=";"`。

> 问自己:> - 如果 CSV 列之间用制表符分隔,sep 应该填什么?> - 文件第 3 行才是列名,前 2 行是说明,应该用哪个参数?> - encoding 不指定,pandas 默认尝试什么编码?

In [ ]:
import pandas as pd# 写出分号分隔的 CSVwith open("semi.csv", "w", encoding="utf-8") as f:    f.write("姓名;年龄;成绩\n")    f.write("张三;20;88.5\n")# ======================# 学员代码区# ======================# 请用 sep 参数正确读取 semi.csv,打印出来pass# ======================# 参考答案# ======================df = pd.read_csv("semi.csv", sep=";", encoding="utf-8")print("--- 分号分隔 ---")print(df)

#### read_excel(Excel 文件读取)痛点:Excel 不像 CSV 是纯文本,它是二进制格式。`pd.read_csv` 读不了.xlsx,必须用 `read_excel`,且常遇到多个 sheet 的情况。类比:Excel 文件像一本活页夹 -- `sheet_name` 选哪一页、`header` 选第几行作列名、`usecols` 只读需要的列。解释:`read_excel` 读取 .xlsx/.xls 文件,核心参数:`io` 文件路径、`sheet_name` 工作表名或索引(默认 0 即第一张)、`header` 列名所在行、`usecols` 只读部分列。

In [ ]:
import pandas as pd# 构造数据写出到 Excel(需要 openpyxl,先 pip install openpyxl)df = pd.DataFrame({    "班级": ["一班", "一班", "二班", "二班"],    "姓名": ["张三", "李四", "王五", "赵六"],    "成绩": [88, 92, 76, 81],})df.to_excel("score.xlsx", index=False, sheet_name="期中成绩")# --- 执行过程 ---# 第 1 行 df = pd.DataFrame({...}):内存创建 DataFrame# 第 2 行 df.to_excel(...):二进制写出 .xlsx 文件,sheet_name 指定工#     作表名df_back = pd.read_excel("score.xlsx", sheet_name="期中成绩")# 第 3 行 pd.read_excel:#   ① 打开 Excel 文件,定位到'期中成绩'工作表#   ② 第 0 行识别为列名print("--- 单 sheet 读取 ---")print(df_back)

> **逐行解剖**> - `sheet_name="期中成绩"`:按名字取工作表。传入整数 0/1/2...按索>   引取。默认 0 即第一个 sheet。> - `header=0`:和第 0 行作为列名。如果 Excel 首行是标题,要设 >   `header=1`。> - `usecols=["班级","成绩"]`:只读需要的列,大文件可节省内存。

> **常见错误**> 1. **错误现象**:`ModuleNotFoundError: No module named 'openpyxl'`>    **原因**:read_excel 需要 openpyxl 库,运行 `pip install openpyxl`>   安装。> 2. **错误现象**:列名变成 "Unnamed: 0" 等>    **原因**:Excel 首行是标题不是列名,应改 `header=1` 跳过首行。

> 问自己:> - 多张 sheet 怎么一次全部读入?> - `usecols` 传入列名列表还是列索引列表?> - xls(旧格式)和 xlsx 用同一个函数读吗?

In [ ]:
import pandas as pd# ======================# 学员代码区# ======================# 读取 score.xlsx(sheet_name="期中成绩"),只取"姓名"与"成绩"两列pass# ======================# 参考答案# ======================df = pd.read_excel(    "score.xlsx",    sheet_name="期中成绩",    usecols=["姓名", "成绩"],)print(df)

#### read_json(JSON 数据读取)痛点:从 API 拿到的 JSON 数据往往是多层嵌套字典,直接 `pd.read_json()`报错 -- 因为 pandas 默认只识别几种固定格式。类比:JSON 像一封信:平铺的列表套字典就是'白话信'(直接读),嵌套的字典套字典就是'加密信'(需要展平)。解释:`read_json` 读取 JSON 文件或字符串; `pd.json_normalize` 展平嵌套结构。`orient` 参数告诉 pandas JSON 的排列方式。

In [ ]:
import pandas as pd# 构造样本数据写出 JSON 文件df = pd.DataFrame({    "姓名": ["张三", "李四"],    "年龄": [20, 21],})df.to_json("students.json", orient="records", force_ascii=False)# --- 执行过程 ---# 第 1 行 df.to_json(..., orient='records', force_ascii=False):#   ① orient='records' 生成 [{'姓名':'张三','年龄':20}, ...]#   ② force_ascii=False 保留中文原样,不转 \uXXXXwith open("students.json", "r", encoding="utf-8") as f:    print("--- JSON 文件内容 ---")    print(f.read()[:80])# 从文件读回 DataFramedf_back = pd.read_json("students.json")print("--- read_json 读回 ---")print(df_back)

> **ASCII 内存图:JSON 嵌套 vs 展平**>> ```> 嵌套结构(data)            展平后(df)> +-------------------+     +----------------------------+> | dict:             |     | DataFrame:                 |> |   "name": "张三"  | --> | 姓名 | 班级 | 成绩.语文    |> |   "score": {      |     | 张三 | 一班 | 88           |> |     "Chinese": 88  |     "                            |> |   }               |     +----------------------------+> +-------------------+> ```

In [ ]:
import pandas as pd# 嵌套 JSON:每个学生的"成绩"字段是子字典data = [    {"姓名": "张三", "班级": "一班",     "成绩": {"语文": 88, "数学": 92}},    {"姓名": "李四", "班级": "二班",     "成绩": {"语文": 75, "数学": 85}},]# --- 执行过程 ---# 第 1 行 data = [...]:内存中嵌套结构# 第 2 行 pd.json_normalize:#   ① 把内层子字典展开成平铺列,列名用 '.' 连接#   ② meta 指定保留的外层字段df = pd.json_normalize(data, meta=["姓名", "班级"])print("--- 展平后 ---")print(df)print("")print("列名:", list(df.columns))

> **逐行解剖**> - `orient="records"`:JSON 是 `[{...},{...}]` 形式,每个元素是一>   条记录,最常见。> - `force_ascii=False`:让中文原样输出,否则变成 \uXXXX 这种转义串。> - `pd.json_normalize(data)`:自动展平内部字典,列名用 `.` 连接层级。

> **常见错误> 1. **错误现象**:`ValueError: Invalid file path or buffer object >    type`>    **原因**:把 Python 列表(嵌套字典)直接传给 read_json,应改用 >   json_normalize。> 2. **错误现象**:JSON 里的中文全变成 \uXXXX>    **原因**:to_json 时漏了 force_ascii=False。

> 问自己:> - orient="records" 和 orient="index" 有什么区别?> - json_normalize 展平后列名长什么样?> - read_json 既能读文件路径,也能读 JSON 字符串吗?

In [ ]:
import pandas as pdjson_str = (    '[{"姓名": "王五", "城市": "深圳"}, '    '{"姓名": "赵六", "城市": "杭州"}]')# ======================# 学员代码区# ======================# 请用 pd.read_json 把 json_str 转成 DataFrame,# 并把"姓名"设为行索引pass# ======================# 参考答案# ======================df = pd.read_json(json_str)df = df.set_index("姓名")print(df)

#### 数据库(SQLite / read_sql)痛点:数据存在 SQLite 数据库文件中,不知道如何用 pandas 读取 -- 必须先连库、写 SQL、再转成 DataFrame。类比:数据库就像一个文件柜:`sqlite3.connect()` 是打开柜子的钥匙、`cursor` 是取文件的双手、`pd.read_sql()` 是把结果整理成表格。解释:标准流程:连接库 -> 执行 SQL -> `read_sql` 把结果转 DataFrame -> **用完关闭连接**。推荐用 `with` 上下文管理器自动关闭。

In [ ]:
import pandas as pdimport sqlite3# 创建内存数据库,退出即丢,适合教学conn = sqlite3.connect(":memory:")cur = conn.cursor()# --- 执行过程 ---# 第 1 行 cur.execute:建表 students,含 id/name/age 三列cur.execute(    "CREATE TABLE students ("    "id INTEGER PRIMARY KEY, name TEXT, age INTEGER)")rows = [    (1, "张三", 20),    (2, "李四", 21),    (3, "王五", 19),    (4, "赵六", 22),]# 第 2 行 executemany:批量插入,? 是占位符防 SQL 注入cur.executemany("INSERT INTO students VALUES (?, ?, ?)", rows)# 第 3 行 conn.commit:提交事务,让插入生效conn.commit()# 第 4 行 pd.read_sql:把 SELECT 结果直接转为 DataFramedf = pd.read_sql("SELECT * FROM students", conn)print("--- 全部数据 ---")print(df)# 第 5 行 带 WHERE 的查询df_where = pd.read_sql(    "SELECT name, age FROM students WHERE age >= 21", conn)print("")print("--- 年龄 >=21 ---")print(df_where)conn.close()

> **逐行解剖**> - `sqlite3.connect(":memory:")`:内存数据库,重启即丢;传文件路径则>   持久化到磁盘。> - `cur.executemany(sql, rows)`:批量执行,`?` 是占位符,防 SQL 注入。> - `conn.commit()`:事务提交,增删改操作必须调才生效。> - `pd.read_sql(sql, conn)`:把 SELECT 结果转 DataFrame,比遍历 cursor >   方便得多。

> **常见错误**> 1. **错误现象**:`sqlite3.OperationalError: no such table: >    students`>    **原因**:还没建表就查表,或数据库路径错,连到空库。> 2. **错误现象**:插入数据后查询结果为空>    **原因**:漏调 `conn.commit()`,事务未提交,数据未落库。

> 问自己:> - `with sqlite3.connect(...) as conn:` 的好处是什么?> - read_sql 能执行 INSERT 吗?还是只能 SELECT?> - 占位符为什么用 `?` 而不是字符串拼接?

In [ ]:
import pandas as pdimport sqlite3# ======================# 学员代码区# ======================# 用 sqlite3 在 :memory: 库中建 books 表,# 列: id INTEGER PRIMARY KEY, title TEXT, price REAL# 然后插入两本书,再用 read_sql 查全部pass# ======================# 参考答案# ======================with sqlite3.connect(":memory:") as conn:    cur = conn.cursor()    cur.execute(        "CREATE TABLE books ("        "id INTEGER PRIMARY KEY, title TEXT, price REAL)"    )    cur.executemany(        "INSERT INTO books VALUES (?, ?, ?)",        [(1, "Python 入门", 59.8), (2, "数据分析实战", 79.0)],    )    conn.commit()    df = pd.read_sql("SELECT * FROM books", conn)    print(df)

#### API 请求(requests.get + json)痛点:很多数据不在文件里,而在 Web 服务的 API 上。不会用 `requests.get`请求数据,就没法拿到股票、天气、新闻等动态数据。类比:API 像餐厅菜单 -- `get` 是'我要点这份'、`post` 是'带我自己材料来'、返回的 JSON 是厨房端上来的菜。解释:核心流程:构造 URL -> `requests.get()` 发起 GET 请求 -> 检查 `status_code` -> `.json()` 把响应解析为 Python 字典或列表。

In [ ]:
import json# 没有真实网络也能教学:用 class 模拟 API 返回# 真实代码会写: resp = requests.get(url); data = resp.json()class FakeResponse:    # 模仿 requests 库的响应对象    status_code = 200    def json(self):        # 这是 API 真正返回的 JSON 结构        return {            "city": "北京",            "temperature": 28,            "forecast": [                {"day": "周一", "high": 30, "low": 22},                {"day": "周二", "high": 31, "low": 23},            ],        }resp = FakeResponse()  # 相当于 resp = requests.get(url)# --- 执行过程 ---# 第 1 行 resp.status_code:HTTP 状态码,200 = 成功,404 = 未找到print('状态码:', resp.status_code)# 第 2 行 resp.json():把 JSON 响应体解析为 Python 字典data = resp.json()print("城市:", data["city"])print("温度:", data["temperature"])# 第 3 行 data['forecast']:取出预报列表,遍历每条for item in data['forecast']:    day = item['day']    low_i = item['low']    high_i = item['high']    print(f'{day}: {low_i}~{high_i} 度')

> **逐行解剖**> - `requests.get(url, params={})`:发起 GET 请求,params 会自动拼到 >   URL 上。> - `resp.status_code`:HTTP 状态码。200 成功、4xx 客户端错(404 未找>   到)、5xx 服务端错。> - `resp.json()`:把响应体 JSON 字符串解析为 Python 字典/列表。> - **真实代码**:需要安装 requests(`pip install requests`)后做网络请求。

> **常见错误**> 1. **错误现象**:`ModuleNotFoundError: No module named 'requests'`>    **原因**:requests 不是标准库,需要 `pip install requests` 安装。> 2. **错误现象**:`json.decoder.JSONDecodeError: Expecting value`>    **原因**:API 返回的不是 JSON(可能报错 HTML),先检查 status_code。

> 问自己:> - GET 和 POST 什么时候用哪个?> - resp.json() 能在 resp.status_code != 200 时调用吗?> - params 参数和把参数写在 URL 里有啥区别?

In [ ]:
import pandas as pd# 模拟 API 返回 JSON 列表(真实代码用 requests.get)def fake_api():    return [        {"姓名": "张三", "部门": "技术", "薪资": 15000},        {"姓名": "李四", "部门": "市场", "薪资": 12000},        {"姓名": "王五", "部门": "技术", "薪资": 18000},    ]# ======================# 学员代码区# ======================# 调用 fake_api() 拿到数据,转成 DataFrame,再筛出"技术"部门pass# ======================# 参考答案# ======================data = fake_api()df = pd.DataFrame(data)df_tech = df[df["部门"] == "技术"]print(df_tech)

#### 编码问题(UnicodeDecodeError 处理)痛点:读取中文 CSV 文件时最常见报错就是 `UnicodeDecodeError`,一搜网上会告诉你'改成 encoding=utf8',结果有时有用有时没用 -- 关键要理解编码本质。类比:编码就像密码本 -- 写出用什么密码本,读出也必须用同一本。用错密码本(编码)就乱码。解释:Python 默认用 utf-8 读写文本,但国内 Windows 导出的 CSV 常见 gbk。处理三策略:1) 试常见编码;2) 用 chardet 自动检测;3) 统一在源头就存 utf-8。

In [ ]:
# ---------- 故意触发 UnicodeDecodeError(BREAK IT) ----------import pandas as pd# 写一份 gbk 编码的文件(模拟 Windows 导出)with open("gbk_demo.csv", "w", encoding="gbk") as f:    f.write("姓名,成绩\n")    f.write("张三,88\n")try:    df = pd.read_csv("gbk_demo.csv")  # 默认 encoding="utf-8"    print("读取成功(不会到这里)")except UnicodeDecodeError as e:    print("--- BREAK IT 触发 UnicodeDecodeError ---")    print("")    print("错误信息:", str(e)[:60])    print("原因:文件是 gbk,默认用 utf-8 读,对不上就报这个错")# --- 正确做法:指定 encoding="gbk" ---df_ok = pd.read_csv("gbk_demo.csv", encoding="gbk")print("")print("--- 指定 encoding=gbk 后 ---")print(df_ok)

> **逐行解剖**> - `open(path, "rb")`:二进制模式读文件,不解释字节,适合先用 >   chardet 检测。> - `chardet.detect(raw)`:返回 `{encoding, confidence, language}`,>   自动猜编码。> - `encoding="gbk"`:常见中文编码,兼容 gb2312 与 gb18030。> - 预防:源头就存 utf-8 最省心。CSV 文件建议用 Notepad / VS Code >   转存为 utf-8。

In [ ]:
# ---------- chardet 自动检测编码(可选方案) ----------# 安装:pip install chardet## import chardet## with open("gbk_demo.csv", "rb") as f:         # 先按二进制读#     raw = f.read()#     result = chardet.detect(raw)               # 检测编码#     print(result)                              # {'encoding': ...}##     df = pd.read_csv(#         "gbk_demo.csv",#         encoding=result["encoding"],            # 按检测结果读#     )print('chardet 是可选方案,注释演示,无需执行')

> **逐行解剖**> - `open(path, "rb")`:二进制模式读文件,不解释字节,适合先用 chardet >   检测。> - `chardet.detect(raw)`:返回 `{encoding, confidence, language}`,自>   动猜编码。> - `encoding="gbk"`:常见中文编码,兼容 gb2312 与 gb18030。> - 预防:源头就存 utf-8 最省心。CSV 文件建议用 Notepad / VS Code >   转存为 utf-8。

> **常见错误**> 1. **乱码但没报错**:文件 gbk,读也用 utf-8,但恰好解析成功,只是中文>   乱码。 **解决**:导出时对调编码,哪种编码写的就用哪种读。> 2. **全部改成 utf-8 仍报错**:文件可能带 BOM(字节顺序标记)。>    **解决**:设 encoding="utf-8-sig",自动去掉 BOM。

> 问自己:> - utf-8 和 gbk 哪个支持更多语言?> - 如何判断一份未知 CSV 是哪种编码?> - "utf-8-sig" 在什么场景下比 "utf-8" 更合适?

In [ ]:
# 编码试错顺序练习模板import pandas as pddef read_csv_try_encodings(path):    """按常见编码顺序试,返回第一个成功的 DataFrame"""    for enc in ["utf-8", "gbk", "utf-8-sig", "gb18030"]:        try:            df = pd.read_csv(path, encoding=enc)            print(f"用 {enc} 读取成功")            return df        except UnicodeDecodeError:            continue    raise ValueError(f"常见编码都失败,文件: {path}")# ======================# 学员代码区# ======================# 调用 read_csv_try_encodings 去读 "gbk_demo.csv",打印结果pass# ======================# 参考答案# ======================df = read_csv_try_encodings("gbk_demo.csv")print(df)

**今日小结**| 概念 | 解决的痛点 ||---|---|| read_csv(path, encoding, sep) | 中文乱码、分隔符不对、列名错位 || read_excel(io, sheet_name, usecols) | Excel 二进制读取、多工作表、列筛选 || read_json / json_normalize | 嵌套 JSON 读不了、中文转义乱码 || sqlite3 + read_sql | 数据库数据直接进 DataFrame || requests.get(.json()) | 从 Web API 动态取数据 || 编码问题(utf-8 / gbk / chardet) | UnicodeDecodeError 与中文乱码 |**更多练习**- 当堂练:course/lesson16/in_class/practice01.py ~ practice06.py- 课后作业:course/lesson16/homework/task01.py ~ task03.py